# 13 — Entity Resolution

**Objetivo:** investigar y resolver de forma determinística/auditable bidder→supplier después del 0% del join directo observado en M1.

El principio es conservador: **unresolved es un resultado válido; un match inventado no lo es**.

## Estrategias permitidas en M2

1. igualdad exacta de código SECOP, si existe overlap;
2. igualdad exacta de NIT normalizado por case/espacios/puntuación, únicamente cuando identifica un solo supplier.

No se elimina dígito de verificación, no se convierte el identificador a entero, no se usa fuzzy name matching y no se usa el resultado futuro de adjudicación para “ayudar” al match.

In [ ]:
import pandas as pd
from argos.common.paths import ProjectPaths
from argos.config.silver import load_silver_spec
from argos.silver.entity_resolution import BUILD_IMPLEMENTATION

paths = ProjectPaths.discover(); paths.ensure_runtime_dirs()
SILVER_SPEC = load_silver_spec(paths.configs / "silver.toml")

for report_name in ["11_validation.parquet", "12_validation.parquet"]:
    frame = pd.read_parquet(paths.reports / "silver" / report_name)
    blocking = frame.loc[frame["blocking"]]
    assert not blocking.empty and blocking["passed"].all(), f"Prerequisite failed: {report_name}"

{
    "implementation": BUILD_IMPLEMENTATION,
    "code_strategy": SILVER_SPEC.entity_resolution.code_strategy_enabled,
    "nit_strategy": SILVER_SPEC.entity_resolution.nit_strategy_enabled,
    "coverage_reference_only": SILVER_SPEC.entity_resolution.report_min_resolved_coverage,
}

In [ ]:
from argos.silver.entity_resolution import build_entity_resolution_silver

result = build_entity_resolution_silver(
    reports_root=paths.reports,
    silver_dir=paths.silver,
    manifest_dir=paths.root / "artifacts" / "manifests",
    silver_spec=SILVER_SPEC,
    temp_directory=paths.root / ".cache" / "duckdb",
    overwrite=True,
)
{name: str(path.relative_to(paths.root)) for name, path in result.artifacts.items()}

## Cobertura y unresolved population

In [ ]:
result.summary

El threshold de coverage configurado es **solo una referencia de reporting**. Si no se alcanza, el check puede aparecer como warning/no bloqueante. No bajamos el estándar de identidad para obtener un porcentaje más alto.

In [ ]:
result.validation

In [ ]:
blocking = result.validation.loc[result.validation["blocking"]]
failed = blocking.loc[~blocking["passed"], "check_id"].tolist()
print("13 ✅ PASS" if result.ready_for_next_notebook else "13 ❌ FAIL")
print(f"Blocking checks: {int(blocking['passed'].sum())}/{len(blocking)}")
print("Manifest:", result.manifest_path)
if failed:
    print("Failed:", ", ".join(failed))
assert result.ready_for_next_notebook, "Notebook 13 remains OPEN; resolve blocking checks before 14."

**Siguiente:** `14_silver_validation.ipynb`.